<a href="https://colab.research.google.com/github/Anandmehata/Anand/blob/main/PITProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q earthaccess h5py numpy pandas fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 85.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.6.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.


In [2]:
import earthaccess
import h5py
import numpy as np
import pandas as pd

print("✓ Libraries loaded")

✓ Libraries loaded


In [3]:
earthaccess.login(strategy="interactive", persist=True)

Enter your Earthdata Login username: Preety_sinha
Enter your Earthdata password: ··········


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/NISAR_Project", exist_ok=True)

Mounted at /content/drive


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  NISAR PIT STABILITY PROJECT — FULL CORRECTED PIPELINE
#  3 granules (HH only) -> Time series -> Mean/Std/CV -> PIT_Class
#  -> Surface-wise analysis -> Graphs -> Results
#
#  CALIBRATION FORMULA (per confirmed flowchart / ISRO NISAR Product
#  Format Document, Ch.3 "Radiometric Calibration of RSLC", K_Beta=1):
#     RSLC (I, Q)
#        -> Power = I^2 + Q^2        (this IS Beta0 directly, no scale
#                                      factor multiplication at this step)
#        -> Sigma0_linear = Beta0 / K_Sigma   (K_Sigma from calibration LUT)
#        -> Sigma0_dB = 10 * log10(Sigma0_linear)
#  scaleFactor from calibrationInformation is NOT applied here — no
#  authoritative source confirms its role in the Beta0/Sigma0 chain,
#  so it is intentionally left out rather than guessed at.
#
#  OTHER FIXES:
#   - Duplicate points removed (old Land18, Water10, Water12 — these
#     had identical coordinates to Land17, Water09, Water11)
#   - Surface label made consistent everywhere: Land / Forest / Water
#   - CV_Reliable flag added (CV is unstable when Mean_Sigma0 ~ 0)
# ════════════════════════════════════════════════════════════════════

import earthaccess
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ───────────────────────────────────────────────────────────────────
#  SETTINGS
# ───────────────────────────────────────────────────────────────────
FOLDER = "/content/drive/MyDrive/NISAR_Project/"   # change if needed
ROI_SIZE = 10
HALF = ROI_SIZE // 2
MAX_DIST_DEG = 0.01
POL = "HH"

# Your 3 actual granules, in time order
GRANULES = {
    "G1": "NISAR_L1_PR_RSLC_005_025_D_090_2005_DHDH_A_20251111T224948_20251111T225023_X05009_N_F_J_001",  # 2025-11-11
    "G2": "NISAR_L1_PR_RSLC_008_025_D_090_2005_SHSH_A_20251217T224950_20251217T225025_X05009_N_F_J_001",  # 2025-12-17
    "G3": "NISAR_L1_PR_RSLC_010_025_D_090_2005_SHSH_A_20260110T224951_20260110T225026_X05009_N_F_J_001",  # 2026-01-10
}
BASE_URL = "https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L1_RSLC_BETA_V1/"

# ── Points (duplicates removed: Land18, Water10, Water12 dropped — ──
# ── they had identical coordinates to Land17, Water09, Water11)    ──
POINTS = [
    # ---------------- LAND (29) ----------------
    {"name":"Land01","lat":-3.362460,"lon":-68.198464},
    {"name":"Land02","lat":-3.366187,"lon":-68.207862},
    {"name":"Land03","lat":-3.333452,"lon":-68.128592},
    {"name":"Land04","lat":-3.327837,"lon":-68.108955},
    {"name":"Land05","lat":-3.372763,"lon":-68.204343},
    {"name":"Land06","lat":-2.657996,"lon":-67.243572},
    {"name":"Land07","lat":-2.863708,"lon":-67.793201},
    {"name":"Land08","lat":-2.870654,"lon":-67.801627},
    {"name":"Land09","lat":-3.380109,"lon":-68.191736},
    {"name":"Land10","lat":-2.832217,"lon":-67.295608},
    {"name":"Land11","lat":-2.810855,"lon":-67.281012},
    {"name":"Land12","lat":-2.804071,"lon":-67.270848},
    {"name":"Land13","lat":-3.389994,"lon":-68.183421},
    {"name":"Land14","lat":-2.630405,"lon":-67.160350},
    {"name":"Land15","lat":-3.078230,"lon":-68.090073},
    {"name":"Land16","lat":-2.747311,"lon":-67.033085},
    {"name":"Land17","lat":-3.394366,"lon":-68.163167},
    {"name":"Land19","lat":-2.754780,"lon":-66.774750},
    {"name":"Land20","lat":-2.759164,"lon":-66.760562},
    {"name":"Land21","lat":-3.413737,"lon":-68.226854},
    {"name":"Land22","lat":-2.733981,"lon":-66.722796},
    {"name":"Land23","lat":-2.567749,"lon":-66.599935},
    {"name":"Land24","lat":-2.616591,"lon":-66.634382},
    {"name":"Land25","lat":-2.729272,"lon":-66.805295},
    {"name":"Land26","lat":-3.409756,"lon":-68.239127},
    {"name":"Land27","lat":-2.662571,"lon":-66.855863},
    {"name":"Land28","lat":-2.669804,"lon":-66.877585},
    {"name":"Land29","lat":-3.408527,"lon":-68.251522},
    {"name":"Land30","lat":-2.884534,"lon":-67.720396},

    # ---------------- FOREST (30) ----------------
    {"name":"Forest1","lat":-3.546246,"lon":-68.402751},
    {"name":"Forest2","lat":-3.463356,"lon":-68.158974},
    {"name":"Forest3","lat":-3.447312,"lon":-68.014315},
    {"name":"Forest4","lat":-3.380459,"lon":-67.856262},
    {"name":"Forest5","lat":-3.193247,"lon":-67.767859},
    {"name":"Forest6","lat":-3.286857,"lon":-67.617842},
    {"name":"Forest7","lat":-3.422424,"lon":-68.106721},
    {"name":"Forest8","lat":-3.467283,"lon":-68.022287},
    {"name":"Forest9","lat":-3.389021,"lon":-67.877349},
    {"name":"Forest10","lat":-2.834775,"lon":-67.925912},
    {"name":"Forest11","lat":-2.846373,"lon":-68.093848},
    {"name":"Forest12","lat":-2.963021,"lon":-67.570344},
    {"name":"Forest13","lat":-2.673463,"lon":-67.714256},
    {"name":"Forest14","lat":-2.633878,"lon":-67.603715},
    {"name":"Forest15","lat":-2.581790,"lon":-67.904052},
    {"name":"Forest16","lat":-2.573456,"lon":-67.755969},
    {"name":"Forest17","lat":-2.583874,"lon":-68.004164},
    {"name":"Forest18","lat":-2.746141,"lon":-67.353840},
    {"name":"Forest19","lat":-2.485738,"lon":-67.129335},
    {"name":"Forest20","lat":-2.518135,"lon":-67.066973},
    {"name":"Forest21","lat":-2.518135,"lon":-66.839974},
    {"name":"Forest22","lat":-2.560500,"lon":-66.786342},
    {"name":"Forest23","lat":-2.580436,"lon":-66.732710},
    {"name":"Forest24","lat":-2.768565,"lon":-66.681573},
    {"name":"Forest25","lat":-2.876944,"lon":-66.859929},
    {"name":"Forest26","lat":-2.305045,"lon":-67.508498},
    {"name":"Forest27","lat":-2.303799,"lon":-67.456114},
    {"name":"Forest28","lat":-2.315015,"lon":-67.372548},
    {"name":"Forest29","lat":-2.528103,"lon":-67.644448},
    {"name":"Forest30","lat":-2.535580,"lon":-67.740487},

    # ---------------- WATER (28) ----------------
    {"name":"Water01","lat":-3.371308,"lon":-68.466632},
    {"name":"Water02","lat":-3.366952,"lon":-68.413480},
    {"name":"Water03","lat":-3.349133,"lon":-68.380954},
    {"name":"Water04","lat":-3.327649,"lon":-67.958476},
    {"name":"Water05","lat":-3.371413,"lon":-68.301271},
    {"name":"Water06","lat":-2.652811,"lon":-67.331859},
    {"name":"Water07","lat":-3.348702,"lon":-68.005163},
    {"name":"Water08","lat":-3.341595,"lon":-68.200334},
    {"name":"Water09","lat":-3.336111,"lon":-68.149866},
    {"name":"Water11","lat":-3.349821,"lon":-68.071245},
    {"name":"Water13","lat":-3.340909,"lon":-67.986101},
    {"name":"Water14","lat":-3.345039,"lon":-67.994141},
    {"name":"Water15","lat":-2.707826,"lon":-66.703523},
    {"name":"Water16","lat":-2.624660,"lon":-67.480432},
    {"name":"Water17","lat":-2.727910,"lon":-66.769224},
    {"name":"Water18","lat":-3.176381,"lon":-67.936319},
    {"name":"Water19","lat":-2.669637,"lon":-66.900343},
    {"name":"Water20","lat":-2.771472,"lon":-66.966043},
    {"name":"Water21","lat":-2.782797,"lon":-67.652794},
    {"name":"Water22","lat":-2.761864,"lon":-67.596885},
    {"name":"Water23","lat":-2.763546,"lon":-67.546032},
    {"name":"Water24","lat":-2.754968,"lon":-67.497031},
    {"name":"Water25","lat":-2.734112,"lon":-67.468910},
    {"name":"Water26","lat":-2.697962,"lon":-67.447494},
    {"name":"Water27","lat":-2.763921,"lon":-67.604912},
    {"name":"Water28","lat":-2.887979,"lon":-67.766645},
    {"name":"Water29","lat":-2.752865,"lon":-67.522180},
    {"name":"Water30","lat":-2.711346,"lon":-67.059022},
]


# ═══════════════════════════════════════════════════════════════════
#  PART 1 — HH BACKSCATTER EXTRACTION (per granule)
# ═══════════════════════════════════════════════════════════════════

earthaccess.login(strategy="interactive", persist=True)
fs = earthaccess.get_fsspec_https_session()

_path_cache = {}

def find_dataset(hf, root, name):
    """Locate a dataset by name, searching progressively wider scopes
    since zeroDopplerTime/slantRange paths vary slightly between
    NISAR product versions."""
    cache_key = (id(hf), root, name)
    if cache_key in _path_cache:
        return _path_cache[cache_key]

    def search(group_path):
        found = {}
        def visitor(path, obj):
            if isinstance(obj, h5py.Dataset) and path.split("/")[-1] == name:
                found["path"] = f"{group_path}/{path}"
        hf[group_path].visititems(visitor)
        return found.get("path")

    candidates = [root, root.rsplit("/", 1)[0], "/science/LSAR/RSLC"]
    for scope in candidates:
        p = search(scope)
        if p:
            _path_cache[cache_key] = p
            return p
    raise KeyError(f"Could not find '{name}' under any of: {candidates}")


def extract_point(hf, pt, granule_name):
    geo = "/science/LSAR/RSLC/metadata/geolocationGrid"
    cal = "/science/LSAR/RSLC/metadata/calibrationInformation"

    lat_grid = hf[f"{geo}/coordinateY"][:]
    lon_grid = hf[f"{geo}/coordinateX"][:]
    inc_grid = hf[f"{geo}/incidenceAngle"][:]
    geo_time = hf[find_dataset(hf, geo, "zeroDopplerTime")][:]
    geo_range = hf[find_dataset(hf, geo, "slantRange")][:]

    if lat_grid.ndim == 3:
        lat_grid, lon_grid, inc_grid = lat_grid[0], lon_grid[0], inc_grid[0]
    n_rows, n_cols = lat_grid.shape

    lat_flat = lat_grid.ravel()
    lon_flat = lon_grid.ravel()
    inc_flat = inc_grid.ravel()
    time_flat = np.tile(geo_time[:, None], (1, n_cols)).ravel()
    range_flat = np.tile(geo_range[None, :], (n_rows, 1)).ravel()

    dist = (lat_flat - pt["lat"])**2 + (lon_flat - pt["lon"])**2
    idx = np.argmin(dist)
    if np.sqrt(dist[idx]) > MAX_DIST_DEG:
        return None

    sigma_lut = hf[f"{cal}/geometry/sigma0"][:]
    lut_time = hf[find_dataset(hf, f"{cal}/geometry", "zeroDopplerTime")][:]
    lut_range = hf[find_dataset(hf, f"{cal}/geometry", "slantRange")][:]

    for freq in ["frequencyA", "frequencyB"]:
        swath = f"/science/LSAR/RSLC/swaths/{freq}"
        slc_path = f"{swath}/{POL}"
        if slc_path not in hf:
            continue

        slc_data = hf[slc_path]
        slc_time = hf[find_dataset(hf, swath, "zeroDopplerTime")][:]
        slc_range = hf[find_dataset(hf, swath, "slantRange")][:]

        row = np.argmin(np.abs(slc_time - time_flat[idx]))
        col = np.argmin(np.abs(slc_range - range_flat[idx]))

        r1, r2 = max(0, row - HALF), min(slc_data.shape[0], row + HALF)
        c1, c2 = max(0, col - HALF), min(slc_data.shape[1], col + HALF)
        patch = slc_data[r1:r2, c1:c2]
        if patch.size == 0:
            continue

        # ── CALIBRATION (per confirmed flowchart, K_Beta = 1) ───
        # Power = I^2 + Q^2 = |complex pixel|^2  -- this IS Beta0
        # directly. No scaleFactor multiplication at this step.
        complex_patch = patch.astype(np.complex64)
        power = np.abs(complex_patch)**2
        mean_power = float(np.mean(power))
        raw_db = 10 * np.log10(mean_power + 1e-10)

        beta0_linear = mean_power
        beta0_db = 10 * np.log10(beta0_linear + 1e-10)

        lut_row = np.argmin(np.abs(lut_time - time_flat[idx]))
        lut_col = np.argmin(np.abs(lut_range - range_flat[idx]))
        ksigma = float(sigma_lut[lut_row, lut_col]) if sigma_lut.ndim == 2 else float(sigma_lut[lut_col])
        sigma0_db = 10 * np.log10(beta0_linear / ksigma) if ksigma > 0 else np.nan

        return {
            "Name": pt["name"], "Lat": pt["lat"], "Lon": pt["lon"],
            "Incidence_deg": round(float(inc_flat[idx]), 4),
            "Raw_dB": round(raw_db, 4),
            "Beta0_dB": round(beta0_db, 4),
            "Sigma0_dB": round(sigma0_db, 4),
            "Frequency": freq, "Granule": granule_name,
        }
    return None


def extract_granule(granule_name):
    url = f"{BASE_URL}{granule_name}/{granule_name}.h5"
    print(f"\nOpening: {granule_name}")
    f_obj = fs.open(url, cache_type="blockcache", block_size=8 * 1024 * 1024)
    hf = h5py.File(f_obj, "r")

    rows = []
    for pt in POINTS:
        row = extract_point(hf, pt, granule_name)
        if row is None:
            print(f"  {pt['name']}: outside usable image area — skipped.")
            continue
        rows.append(row)
        print(f"  {pt['name']} ✓  Sigma0={row['Sigma0_dB']} dB")
    hf.close()
    return pd.DataFrame(rows)


# Run extraction for all 3 granules
granule_dfs = {}
for tag, gname in GRANULES.items():
    df_g = extract_granule(gname)
    df_g.to_csv(f"{FOLDER}backscatter_HH_results_{tag}.csv", index=False)
    granule_dfs[tag] = df_g
    print(f"✅ Saved backscatter_HH_results_{tag}.csv ({len(df_g)} points)")


# ═══════════════════════════════════════════════════════════════════
#  PART 2 — MERGE INTO TIME-SERIES DATABASE
# ═══════════════════════════════════════════════════════════════════

g1 = granule_dfs["G1"][["Name", "Lat", "Lon", "Incidence_deg", "Sigma0_dB"]].rename(columns={"Sigma0_dB": "Sigma0_G1"})
g2 = granule_dfs["G2"][["Name", "Sigma0_dB"]].rename(columns={"Sigma0_dB": "Sigma0_G2"})
g3 = granule_dfs["G3"][["Name", "Sigma0_dB"]].rename(columns={"Sigma0_dB": "Sigma0_G3"})

merged = pd.merge(g1, g2, on="Name").merge(g3, on="Name")
merged.to_csv(f"{FOLDER}HH_TimeSeries.csv", index=False)
print(f"\n✅ HH_TimeSeries.csv saved successfully ({len(merged)} points)")


# ═══════════════════════════════════════════════════════════════════
#  PART 3 — MEAN, STD, CV CALCULATION
# ═══════════════════════════════════════════════════════════════════

df = merged.copy()
sigma_cols = ["Sigma0_G1", "Sigma0_G2", "Sigma0_G3"]
df["Mean_Sigma0"] = df[sigma_cols].mean(axis=1)
df["Std_Sigma0"] = df[sigma_cols].std(axis=1, ddof=1)
df["CV (%)"] = (df["Std_Sigma0"] / df["Mean_Sigma0"]).abs() * 100

# Flag points where CV is unreliable (Mean_Sigma0 near zero)
CV_THRESHOLD = 0.1
df["CV_Reliable"] = df["Mean_Sigma0"].abs() >= CV_THRESHOLD

df.to_csv(f"{FOLDER}PIT_Statistics.csv", index=False)
print("✅ PIT_Statistics.csv saved successfully")


# ═══════════════════════════════════════════════════════════════════
#  PART 4 — PIT STABILITY CLASSIFICATION (based on Std_Sigma0)
# ═══════════════════════════════════════════════════════════════════

def classify(std):
    if std <= 0.5:
        return "Excellent PIT"
    elif std <= 1.0:
        return "Good PIT"
    elif std <= 2.0:
        return "Moderately Stable"
    else:
        return "Unstable"

df["PIT_Class"] = df["Std_Sigma0"].apply(classify)
print("\nPIT Class counts:")
print(df["PIT_Class"].value_counts())

# Consistent surface labeling (Land / Forest / Water — no "Bare Land")
def get_surface(name):
    if name.startswith("Water"):
        return "Water"
    elif name.startswith("Forest"):
        return "Forest"
    elif name.startswith("Land"):
        return "Land"
    return "Unknown"

df["Surface"] = df["Name"].apply(get_surface)

df.to_csv(f"{FOLDER}PIT_Stability.csv", index=False)
print("✅ PIT_Stability.csv saved successfully")


# ═══════════════════════════════════════════════════════════════════
#  PART 5 — SURFACE-WISE ANALYSIS
# ═══════════════════════════════════════════════════════════════════

surface_summary = df.groupby("Surface").agg(
    Count=("Name", "count"),
    Mean_Sigma0_avg=("Mean_Sigma0", "mean"),
    Std_Sigma0_avg=("Std_Sigma0", "mean"),
    CV_avg=("CV (%)", "mean")
).round(3)
print("\nPer-Surface Summary:")
print(surface_summary)

pit_counts = pd.crosstab(df["Surface"], df["PIT_Class"])
print("\nPIT Class counts by Surface:")
print(pit_counts)

surface_summary.to_csv(f"{FOLDER}Surfacewise_Summary.csv")
pit_counts.to_csv(f"{FOLDER}Surface_PIT_Count.csv")
print("✅ Surfacewise_Summary.csv and Surface_PIT_Count.csv saved successfully")





Opening: NISAR_L1_PR_RSLC_005_025_D_090_2005_DHDH_A_20251111T224948_20251111T225023_X05009_N_F_J_001
  Land01 ✓  Sigma0=-6.8037 dB
  Land02 ✓  Sigma0=-4.911 dB
  Land03 ✓  Sigma0=-5.404 dB
  Land04 ✓  Sigma0=-6.1756 dB
  Land05 ✓  Sigma0=-6.5572 dB
  Land06 ✓  Sigma0=-9.7368 dB
  Land07 ✓  Sigma0=-9.8423 dB
  Land08 ✓  Sigma0=-6.7636 dB
  Land09 ✓  Sigma0=-6.7274 dB
  Land10 ✓  Sigma0=-6.0194 dB
  Land11 ✓  Sigma0=-8.7552 dB
  Land12 ✓  Sigma0=-5.1856 dB
  Land13 ✓  Sigma0=-6.5293 dB
  Land14 ✓  Sigma0=-5.6263 dB
  Land15 ✓  Sigma0=-6.0113 dB
  Land16 ✓  Sigma0=-6.8148 dB
  Land17 ✓  Sigma0=-6.2388 dB
  Land19 ✓  Sigma0=-5.7463 dB
  Land20 ✓  Sigma0=-7.5274 dB
  Land21 ✓  Sigma0=-3.9734 dB
  Land22 ✓  Sigma0=-5.7977 dB
  Land23 ✓  Sigma0=-7.0535 dB
  Land24 ✓  Sigma0=-10.1404 dB
  Land25 ✓  Sigma0=-8.3384 dB
  Land26 ✓  Sigma0=-4.845 dB
  Land27 ✓  Sigma0=-6.9836 dB
  Land28 ✓  Sigma0=-6.7117 dB
  Land29 ✓  Sigma0=-6.8034 dB
  Land30 ✓  Sigma0=-4.8436 dB
  Forest1 ✓  Sigma0=-6.0012 dB